# 🔬 WaveSemiNet — Semiconductor Image Restoration
**SEMICON India Hackathon 2026** | Wavelet-Guided Dual-Branch Restoration Network

> **Setup:** Go to `Runtime → Change runtime type → T4 GPU` before running


## 1. Setup & Installation

In [ ]:
# Verify GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
else:
    raise RuntimeError("No GPU! Go to Runtime > Change runtime type > T4 GPU")


In [ ]:
# Clone repo and install deps
!git clone https://github.com/theasynch/I4C-SEMICON-India-Hackathon.git
%cd I4C-SEMICON-India-Hackathon
!pip install -q einops lpips PyWavelets albumentations timm torchmetrics scikit-image tensorboard onnx onnxruntime


## 2. Dataset Setup
The data is in `.gitignore` so we need to upload it. Choose one option below.

In [ ]:
import os

# Check if data already exists
train_ok = os.path.exists("Data-public/train/train/NoisyLR")
test_ok = os.path.exists("Data-public/test/NoisyLR")

if train_ok and test_ok:
    print("Dataset already present!")
else:
    print("Dataset not found. Run one of the upload cells below.")


In [ ]:
# OPTION A: Google Drive (recommended for ~900MB files)
# 1. Upload train.zip and Test_NoisyLR.zip to your Google Drive root
# 2. Uncomment and run:

# from google.colab import drive
# drive.mount("/content/drive")
# !cp "/content/drive/MyDrive/train.zip" Data-public/
# !cp "/content/drive/MyDrive/Test_NoisyLR.zip" Data-public/
# !cd Data-public && unzip -q train.zip -d train/ && unzip -q Test_NoisyLR.zip -d test/
# print("Dataset extracted!")


In [ ]:
# OPTION B: Direct browser upload

# from google.colab import files
# import zipfile
# print("Upload train.zip:")
# uploaded = files.upload()
# os.makedirs("Data-public/train", exist_ok=True)
# with zipfile.ZipFile("train.zip", "r") as z:
#     z.extractall("Data-public/train/")
# print("Upload Test_NoisyLR.zip:")
# uploaded = files.upload()
# os.makedirs("Data-public/test", exist_ok=True)
# with zipfile.ZipFile("Test_NoisyLR.zip", "r") as z:
#     z.extractall("Data-public/test/")


In [ ]:
# Verify dataset
train_noisy = os.listdir("Data-public/train/train/NoisyLR")
train_gt = os.listdir("Data-public/train/train/GT")
test_noisy = os.listdir("Data-public/test/NoisyLR")
print(f"Train NoisyLR: {len(train_noisy)} files")
print(f"Train GT:      {len(train_gt)} files")
print(f"Test NoisyLR:  {len(test_noisy)} files")
assert len(train_noisy) == len(train_gt), "Train/GT count mismatch!"
print("Dataset OK!")


## 3. Quick Smoke Test

In [ ]:
from models.waveseminet import build_waveseminet
import yaml

with open("configs/train_unified.yaml", "r") as f:
    config = yaml.safe_load(f)

model = build_waveseminet(config).cuda()
x = torch.randn(1, 1, 128, 128, device="cuda")
with torch.no_grad():
    y = model(x, task_id=0)
print(f"Model: {model.count_parameters():,} params")
print(f"Input: {x.shape} -> Output: {y.shape}")
for name, count in model.get_branch_params().items():
    print(f"  {name}: {count:,}")
del model, x, y; torch.cuda.empty_cache()


## 4. Train! (~2-3 hours on T4)

In [ ]:
!python train.py --config configs/train_unified.yaml --gpu 0


## 5. Evaluate on Test Set

In [ ]:
!python evaluate.py \
    --weights weights/best.pth \
    --data Data-public/test/NoisyLR \
    --config configs/train_unified.yaml \
    --output results/ \
    --save_viz \
    --num_viz 20


## 6. Visualize Results

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

viz_dir = Path("results/visualizations")
viz_files = sorted(viz_dir.glob("*.png"))[:6]
if viz_files:
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    for ax, vf in zip(axes.flat, viz_files):
        ax.imshow(mpimg.imread(str(vf)))
        ax.set_title(vf.stem.replace("_comparison", ""), fontsize=10)
        ax.axis("off")
    plt.suptitle("WaveSemiNet Restoration Results", fontsize=16, fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print("No visualizations yet. Run evaluation first.")


## 7. Export ONNX (Optional)

In [ ]:
!python scripts/export_onnx.py \
    --weights weights/best.pth \
    --config configs/train_unified.yaml \
    --output weights/waveseminet.onnx


## 8. Download Trained Weights & Results

In [ ]:
!zip -r waveseminet_results.zip weights/ results/
from google.colab import files
files.download("waveseminet_results.zip")
print("Download started! Contains trained weights + restoration results.")
